# Notebook 13 – Preventing Data Leakage

## 1. What Is Data Leakage?
Data leakage happens when information unavailable for legitimate training or prediction reaches the model, often producing misleadingly high performance.

## 2. Types
- **Target leakage:** a feature reveals the answer or is created after the outcome.
- **Train-test contamination:** test information influences training.
- **Feature leakage:** a feature indirectly reveals the target.
- **Preprocessing leakage:** preprocessing is fitted on all data before splitting.
- **Temporal leakage:** future information is used to predict the past.

## 3. Example: Customer Churn
A field such as `cancellation_confirmed_after_call` may only be known after a customer churns, so it should not be used to predict churn beforehand.

In [1]:
import pandas as pd

df = pd.DataFrame({
    "tenure_months": [2, 5, 12, 18, 24, 36, 8, 30],
    "monthly_charges": [80, 70, 65, 60, 55, 50, 90, 45],
    "support_calls": [5, 4, 3, 2, 1, 0, 6, 1],
    "cancellation_confirmed_after_call": [1, 1, 0, 0, 0, 0, 1, 0],
    "Churn": [1, 1, 0, 0, 0, 0, 1, 0]
})
df

,tenure_months,monthly_charges,support_calls,cancellation_confirmed_after_call,Churn
0,2,80,5,1,1
1,5,70,4,1,1
2,12,65,3,0,0
3,18,60,2,0,0
4,24,55,1,0,0
5,36,50,0,0,0
6,8,90,6,1,1
7,30,45,1,0,0


## 4. Target / Feature Leakage

### Incorrect workflow
Include a feature that directly reveals the outcome or is only recorded after it happens. The model may appear excellent but cannot use that field at prediction time.

In [2]:
# INCORRECT
X_bad = df[["tenure_months", "monthly_charges", "support_calls",
            "cancellation_confirmed_after_call"]]
y = df["Churn"]
print("Suspicious feature included:", "cancellation_confirmed_after_call" in X_bad.columns)

Suspicious feature included: True


### Correct workflow
Use only fields available at the time the prediction is made.

In [3]:
# CORRECT
X_good = df[["tenure_months", "monthly_charges", "support_calls"]]
y = df["Churn"]
print("Features used:", X_good.columns.tolist())

Features used: ['tenure_months', 'monthly_charges', 'support_calls']


## 5. Preprocessing Leakage

### Incorrect workflow
Fit a scaler before splitting. The scaling parameters use both training and test data.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X_good, y, test_size=0.25, random_state=42, stratify=y
)
# INCORRECT: fits on all rows
scaler_bad = StandardScaler()
X_all_scaled_bad = scaler_bad.fit_transform(X_good)
print("Incorrect: fit_transform used on the full dataset.")

Incorrect: fit_transform used on the full dataset.


### Correct workflow
Fit the scaler on training data only; transform test data without refitting.

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Train shape: (6, 3)
Test shape: (2, 3)


## 6. Pipeline-Based Prevention
A pipeline fits preprocessing as part of model training. With cross-validation, each fold learns its preprocessing from that fold's training portion.

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression

pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
pipeline.fit(X_train, y_train)
print("Pipeline fitted successfully.")

Pipeline fitted successfully.


## 7. Train-Test Contamination
Do not repeatedly tune features, thresholds, or hyperparameters based on test scores. Use validation data or cross-validation for choices and keep the test set for final evaluation.

## 8. Temporal Leakage
For time-based prediction, features must be available at the prediction timestamp. Sort by date and train on the past, then evaluate on the future.

In [7]:
time_df = pd.DataFrame({
    "date": pd.date_range("2026-01-01", periods=6, freq="MS"),
    "sales": [10, 12, 11, 15, 14, 18]
}).sort_values("date")

cut = 4
train_time = time_df.iloc[:cut]
test_time = time_df.iloc[cut:]
print("Train dates:", train_time["date"].min().date(), "to", train_time["date"].max().date())
print("Test dates:", test_time["date"].min().date(), "to", test_time["date"].max().date())

Train dates: 2026-01-01 to 2026-04-01
Test dates: 2026-05-01 to 2026-06-01


## 9. How to Detect Leakage
- Ask whether every feature exists at prediction time.
- Check for target-derived or post-outcome columns.
- Investigate unusually high or perfect model scores.
- Audit joins, duplicates, aggregations, and feature engineering.
- Compare with time-aware validation when appropriate.

## 10. How to Prevent Leakage
1. Define the target and prediction timestamp.
2. Split before fitting preprocessing.
3. Use pipelines and cross-validation.
4. Keep the test set untouched until final evaluation.
5. Use chronological splits for time-based tasks.
6. Remove target-derived and post-outcome features.
7. Fit imputation, scaling, encoding, feature selection, and resampling on training data only.

## Final Takeaway
A high score is trustworthy only when the evaluation process mirrors how the model will be used in the real world.